# Cross-Validation — TrOCR checkpoint-5750

Loads the best checkpoint (step 5750, epoch 10, CER=0.0399) and its optimizer state,  
then evaluates CER/WER across **5 stratified folds** of the full Bentham dataset.

In [ ]:
import os
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torchvision.transforms as transforms
import evaluate

from PIL import Image
from dataclasses import dataclass
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import KFold
from transformers import TrOCRProcessor, VisionEncoderDecoderModel

plt.rcParams["figure.figsize"] = (12, 4)

In [ ]:
def seed_everything(seed=42):
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seed_everything(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

## Paths

In [ ]:
@dataclass(frozen=True)
class DatasetConfig:
    IMAGES_DIR:         str = "../data/Bentham/BenthamDatasetR0-GT/Images/Lines"
    TRANSCRIPTIONS_DIR: str = "../data/Bentham/BenthamDatasetR0-GT/Transcriptions"
    TRAIN_LST:          str = "../data/Bentham/BenthamDatasetR0-GT/Partitions/TrainLines.lst"
    VAL_LST:            str = "../data/Bentham/BenthamDatasetR0-GT/Partitions/ValidationLines.lst"
    TEST_LST:           str = "../data/Bentham/BenthamDatasetR0-GT/Partitions/TestLines.lst"
    MAX_TARGET_LENGTH:  int = 128

# checkpoint-5750 is the best/final model (epoch 10, val CER=0.0399)
CHECKPOINT_DIR = "../models/bentham_trocr_checkpoint/checkpoint-5750-20260612T084901Z-3-001/checkpoint-5750"
OPTIMIZER_PATH = "../models/bentham_trocr_checkpoint/optimizer-002.pt"

N_FOLDS     = 5
BATCH_SIZE  = 8
NUM_BEAMS   = 4

## Load Model & Optimizer from Checkpoint

In [ ]:
processor = TrOCRProcessor.from_pretrained(CHECKPOINT_DIR)
model     = VisionEncoderDecoderModel.from_pretrained(CHECKPOINT_DIR)
model.to(device)
model.eval()
print("Model loaded from", CHECKPOINT_DIR)

In [ ]:
# Load the optimizer state saved alongside the checkpoint.
# Only required when resuming training; included here for completeness.
optimizer_state = torch.load(OPTIMIZER_PATH, map_location="cpu")
print("Optimizer state loaded from", OPTIMIZER_PATH)
print("  Keys:", list(optimizer_state.keys()) if isinstance(optimizer_state, dict) else type(optimizer_state))

## Dataset Setup

In [ ]:
def build_dataframe(partition_lst, images_dir, transcriptions_dir):
    with open(partition_lst) as f:
        ids = [line.strip() for line in f if line.strip()]
    rows, skipped = [], 0
    for stem in ids:
        img_path = os.path.join(images_dir, stem + ".png")
        txt_path = os.path.join(transcriptions_dir, stem + ".txt")
        if not os.path.exists(img_path) or not os.path.exists(txt_path):
            skipped += 1
            continue
        with open(txt_path, encoding="utf-8") as f:
            text = f.read().strip()
        if not text:
            skipped += 1
            continue
        rows.append({"file_name": img_path, "text": text})
    df = pd.DataFrame(rows).reset_index(drop=True)
    print(f"  Loaded {len(df)} samples  (skipped {skipped})")
    return df


print("Building splits...")
train_df = build_dataframe(DatasetConfig.TRAIN_LST, DatasetConfig.IMAGES_DIR, DatasetConfig.TRANSCRIPTIONS_DIR)
val_df   = build_dataframe(DatasetConfig.VAL_LST,   DatasetConfig.IMAGES_DIR, DatasetConfig.TRANSCRIPTIONS_DIR)
test_df  = build_dataframe(DatasetConfig.TEST_LST,  DatasetConfig.IMAGES_DIR, DatasetConfig.TRANSCRIPTIONS_DIR)

all_df = pd.concat([train_df, val_df, test_df], ignore_index=True)
print(f"\nCombined: {len(all_df)} samples  (train {len(train_df)} + val {len(val_df)} + test {len(test_df)})")

In [ ]:
eval_transforms = transforms.Compose([
    transforms.Grayscale(num_output_channels=3),
])


class BenthamOCRDataset(Dataset):
    def __init__(self, df, processor, transforms=None, max_target_length=128):
        self.df = df.reset_index(drop=True)
        self.processor = processor
        self.transforms = transforms
        self.max_target_length = max_target_length

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        image = Image.open(self.df["file_name"][idx]).convert("RGB")
        if self.transforms:
            image = self.transforms(image)
        pixel_values = self.processor(image, return_tensors="pt").pixel_values
        return {
            "pixel_values": pixel_values.squeeze(),
            "text": self.df["text"][idx],
        }

## Metrics

In [ ]:
cer_metric = evaluate.load("cer")
wer_metric = evaluate.load("wer")

## 5-Fold Cross-Validation

Evaluate the frozen checkpoint-5750 weights on each held-out fold.  
No retraining — this measures performance variance across the full dataset.

In [ ]:
@torch.no_grad()
def evaluate_fold(fold_df):
    """Return (cer, wer, pred_strs, label_strs) for a single fold."""
    dataset = BenthamOCRDataset(fold_df, processor, eval_transforms, DatasetConfig.MAX_TARGET_LENGTH)
    loader  = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

    pred_strs, label_strs = [], []
    for batch_idx, batch in enumerate(loader):
        pixel_values = batch["pixel_values"].to(device)
        generated_ids = model.generate(
            pixel_values,
            num_beams=NUM_BEAMS,
            max_length=DatasetConfig.MAX_TARGET_LENGTH,
        )
        preds  = processor.batch_decode(generated_ids, skip_special_tokens=True)
        labels = list(batch["text"])
        pred_strs.extend(preds)
        label_strs.extend(labels)

        if (batch_idx + 1) % 20 == 0:
            done = (batch_idx + 1) * BATCH_SIZE
            print(f"    {min(done, len(fold_df))}/{len(fold_df)} samples processed", end="\r")

    print()
    cer = cer_metric.compute(predictions=pred_strs, references=label_strs)
    wer = wer_metric.compute(predictions=pred_strs, references=label_strs)
    return cer, wer, pred_strs, label_strs

In [ ]:
kf = KFold(n_splits=N_FOLDS, shuffle=True, random_state=42)
fold_results = []

for fold, (_, test_idx) in enumerate(kf.split(all_df)):
    fold_df = all_df.iloc[test_idx].reset_index(drop=True)
    print(f"\nFold {fold + 1}/{N_FOLDS}  ({len(fold_df)} samples)")

    cer, wer, preds, labels = evaluate_fold(fold_df)
    fold_results.append({"fold": fold + 1, "n": len(fold_df), "cer": cer, "wer": wer})
    print(f"  CER: {cer:.4f}   WER: {wer:.4f}")

## Results

In [ ]:
results_df = pd.DataFrame(fold_results)

summary = pd.concat([
    results_df,
    pd.DataFrame([{
        "fold": "mean ± std",
        "n": results_df["n"].sum(),
        "cer": f"{results_df['cer'].mean():.4f} ± {results_df['cer'].std():.4f}",
        "wer": f"{results_df['wer'].mean():.4f} ± {results_df['wer'].std():.4f}",
    }])
], ignore_index=True)

print(summary.to_string(index=False))

In [ ]:
folds = results_df["fold"]
cers  = results_df["cer"]
wers  = results_df["wer"]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].bar(folds, cers, color="steelblue", alpha=0.8)
axes[0].axhline(cers.mean(), color="red", linestyle="--", label=f"mean={cers.mean():.4f}")
axes[0].set_title("CER per Fold")
axes[0].set_xlabel("Fold")
axes[0].set_ylabel("CER")
axes[0].legend()

axes[1].bar(folds, wers, color="coral", alpha=0.8)
axes[1].axhline(wers.mean(), color="red", linestyle="--", label=f"mean={wers.mean():.4f}")
axes[1].set_title("WER per Fold")
axes[1].set_xlabel("Fold")
axes[1].set_ylabel("WER")
axes[1].legend()

plt.suptitle("5-Fold Cross-Validation — checkpoint-5750", fontsize=13)
plt.tight_layout()
plt.show()